# Volume 4, Chapter 4: Integration Techniques (Python companion)

Exercises first, solutions at the end. The chapter computed integrals exactly; here we ask how a computer computes them numerically, why it sometimes fails silently, and how to check every integral of the chapter to fifty digits. Requirements: numpy, mpmath and sympy.

*Before each solution, predict what the output will look like.*

## Exercises

**P4.1 (★) Three quadrature rules.** Compute $\int_0^1e^x\dd x$ with the trapezoid rule, Simpson's rule and Gauss–Legendre quadrature with $n=2,4,\dots,32$ points, and read off how fast each error decreases. Then apply the trapezoid rule to the periodic function $e^{\cos x}$ on $[0,2\pi]$: what changes?

**P4.2 (★) Fifty digits.** Check every closed form of the chapter to $50$ digits with mpmath's `quad` (tanh-sinh quadrature): Example 4.6, the Gaussian integral, two instances of Frullani's Theorem 4.8, Proposition 4.9, and Exercises 4.4, 4.5 and 4.6.

**P4.3 (★★) The Dirichlet integral, numerically.** Try to compute $\int_0^\infty\frac{\sin x}x\dd x$ (Theorem 4.11) by truncating at $X=10,100,1000$, and by calling `quad` on $[0,\infty[$ directly. Then do it properly, by splitting into the lobes $[k\pi,(k+1)\pi]$ and accelerating the alternating series of lobes.

**P4.4 (★) Feynman's trick by computer algebra.** Let sympy differentiate the integrand of Exercise 4.6 in $a$, split it into partial fractions and integrate it. Do the same for Example 4.6.

**P4.5 (★★) CS application: adaptive quadrature.** Write adaptive Simpson quadrature: estimate the integral on $[a,b]$ with one Simpson step and with two half steps; if they agree to within $15\,\mathrm{tol}$, accept, otherwise recurse on both halves with $\mathrm{tol}/2$. Count function evaluations for $\int_0^1\sqrt x\dd x$ and compare with uniform Simpson at the same accuracy.

**P4.6 (★★) The curse of dimension.** Integrate $\prod_{i=1}^d\frac\pi2\sin(\pi x_i)$ over $[0,1]^d$ (exact value $1$, by Fubini) with $10^5$ points, once by Monte Carlo and once by a midpoint grid, for $d=1,2,5,10$.

## Solutions

### P4.1

In [ ]:
import math
import numpy as np

def trapezoid(f, a, b, n):
    x = np.linspace(a, b, n + 1); y = f(x)
    return (b - a) / n * (y.sum() - (y[0] + y[-1]) / 2)

def simpson(f, a, b, n):          # n even
    x = np.linspace(a, b, n + 1); y = f(x)
    return (b - a) / (3 * n) * (y[0] + y[-1] + 4 * y[1:-1:2].sum() + 2 * y[2:-1:2].sum())

def gauss(f, a, b, n):
    x, w = np.polynomial.legendre.leggauss(n)
    return (b - a) / 2 * (w * f((b - a) / 2 * x + (a + b) / 2)).sum()

exact = math.e - 1
print(" n   trapezoid   Simpson     Gauss-Legendre")
for n in (2, 4, 8, 16, 32):
    print(f"{n:2d}  {abs(trapezoid(np.exp, 0, 1, n) - exact):.1e}   {abs(simpson(np.exp, 0, 1, n) - exact):.1e}   "
          f"{abs(gauss(np.exp, 0, 1, n) - exact):.1e}")

The trapezoid error is divided by $4$ when $n$ doubles (order $n^{-2}$), Simpson's by $16$ (order $n^{-4}$), and Gauss–Legendre reaches machine precision with $8$ points: with $n$ well-chosen nodes and weights it is exact for polynomials of degree $2n-1$, so its error decreases faster than any power of $n$ for a function as smooth as $e^x$.

In [ ]:
from mpmath import besseli
f = lambda x: np.exp(np.cos(x))          # periodic and smooth
exact = 2 * math.pi * float(besseli(0, 1))     # 2 pi I_0(1)
print("periodic integrand e^cos x on [0, 2 pi]:")
for n in (2, 4, 8, 16):
    print(f"  trapezoid with {n:2d} points: error {abs(trapezoid(f, 0, 2 * math.pi, n) - exact):.1e}")

For a smooth periodic function the humble trapezoid rule converges exponentially fast: $16$ points give machine precision. The error terms of the trapezoid rule involve only the values of derivatives at the ends of the interval (this is the Euler–Maclaurin formula of Chapter 11), and for a periodic function they cancel. Chapter 6 gives a second explanation: the error is a sum of Fourier coefficients, which decay exponentially for analytic functions.

### P4.2

In [ ]:
from mpmath import mp, quad, log, exp, sin, cos, pi, inf, sqrt, mpf, atan
mp.dps = 50
checks = [
    ("Example logint    ", quad(lambda x: (x - 1) / log(x), [0, 1]), log(2)),
    ("Gaussian          ", quad(lambda x: exp(-x**2), [-inf, inf]), sqrt(pi)),
    ("Frullani e^-x     ", quad(lambda x: (exp(-2*x) - exp(-5*x)) / x, [0, inf]), log(mpf(5) / 2)),
    ("Frullani arctan   ", quad(lambda x: (atan(3*x) - atan(x)) / x, [0, 1, inf]), (pi / 2) * log(3)),
    ("log sin           ", quad(lambda x: log(sin(x)), [0, pi / 2]), -pi / 2 * log(2)),
    ("Exercise xsinx    ", quad(lambda x: x * sin(x) / (1 + cos(x)**2), [0, pi]), pi**2 / 4),
    ("Exercise loglog   ", quad(lambda x: log(x) * log(1 - x), [0, 1]), 2 - pi**2 / 6),
    ("Exercise arctan a=2", quad(lambda x: log(1 + 4*x**2) / (1 + x**2), [0, 1, inf]), pi * log(3)),
]
for name, got, want in checks:
    print(f"{name}  {mp.nstr(got, 20)}   difference {mp.nstr(abs(got - want), 3)}")

Every closed form holds to $50$ digits. Tanh-sinh quadrature substitutes $x=\tanh(\frac\pi2\sinh u)$, which makes the integrand decay doubly exponentially at both ends of the interval; endpoint singularities such as $\ln x$ at $0$ then cost nothing. This is why mpmath's default handles $\ln\sin x$ and $\ln x\ln(1-x)$ effortlessly. A numerical match is not a proof, but a mismatch would have been a bug in the book.

### P4.3

In [ ]:
from mpmath import quadosc, si
mp.dps = 30
naive = [quad(lambda x: sin(x) / x, [0, X]) for X in (10, 100, 1000)]
print("Si(10), Si(100), Si(1000):", [mp.nstr(v, 10) for v in naive])
print("quad on [0, inf] directly:", mp.nstr(quad(lambda x: sin(x) / x, [0, inf]), 15))
print("quadosc                  :", mp.nstr(quadosc(lambda x: sin(x) / x, [0, inf], omega=1), 25))
print("pi / 2                   :", mp.nstr(pi / 2, 25))

Truncation converges slowly, like $\frac{\cos X}X$ by an integration by parts, so $X=1000$ gives only three digits. Worse, `quad` on $[0,\infty[$ returns $5.10$ without any warning: the integrand is not absolutely integrable, and a method that samples it at a few points cannot see infinitely many oscillations. This is the numerical shadow of the paragraph before Theorem 4.11: $\frac{\sin x}x$ is not integrable, and the integral exists only as a limit. `quadosc` succeeds because it knows the period.

In [ ]:
from mpmath import nsum
lobes = lambda k: quad(lambda x: sin(x) / x, [k * pi, (k + 1) * pi])
print("first lobes:", [mp.nstr(lobes(k), 6) for k in range(5)])
print("sum of lobes, accelerated:", mp.nstr(nsum(lobes, [0, inf]), 25))
print("Gibbs: Si(pi) =", mp.nstr(si(pi), 12), " overshoot", mp.nstr(si(pi) / (pi / 2) - 1, 6))

The lobes alternate in sign and decrease, so the integral is the sum of an alternating series (by the alternating series test of Volume 0), which acceleration (Chapter 3 notebook) sums to full precision. The first lobe alone is $\mathrm{Si}(\pi)\approx1.852$, $18\%$ above $\frac\pi2$, as in the figure of the chapter.

### P4.4

In [ ]:
import sympy as sp
x, a = sp.symbols('x a', positive=True)
g = sp.diff(sp.log(1 + a**2 * x**2) / (1 + x**2), a)
print("integrand of J'(a):", sp.simplify(g))
print("partial fractions :", sp.apart(sp.simplify(g), x))
Jp = sp.integrate(sp.apart(sp.simplify(g), x), (x, 0, sp.oo))
print("J'(a) =", sp.simplify(Jp))
t = sp.symbols('t', nonnegative=True)
print("F'(t) for (x^t - 1)/ln x:", sp.integrate(x**t, (x, 0, 1)))

Sympy reproduces the partial fractions of the solution and finds $J'(a)=\frac\pi{1+a}$. Notice what the computer does *not* check: that differentiating under the integral sign was allowed. That is the content of Theorem 4.4, and it is exactly where human work remains.

### P4.5

In [ ]:
calls = 0
def counted(f):
    def h(x):
        global calls
        calls += 1
        return f(x)
    return h

def adaptive_simpson(f, a, b, tol):
    def simp(a, fa, b, fb):
        m = (a + b) / 2; fm = f(m)
        return m, fm, (b - a) / 6 * (fa + 4 * fm + fb)
    def rec(a, fa, b, fb, m, fm, whole, tol, depth):
        lm, flm, left = simp(a, fa, m, fm)
        rm, frm, right = simp(m, fm, b, fb)
        if depth > 50 or abs(left + right - whole) <= 15 * tol:
            return left + right + (left + right - whole) / 15
        return (rec(a, fa, m, fm, lm, flm, left, tol / 2, depth + 1) +
                rec(m, fm, b, fb, rm, frm, right, tol / 2, depth + 1))
    fa, fb = f(a), f(b)
    m, fm, whole = simp(a, fa, b, fb)
    return rec(a, fa, b, fb, m, fm, whole, tol, 0)

f = lambda x: math.sqrt(x)            # derivative blows up at 0
for tol in (1e-4, 1e-7, 1e-10):
    calls = 0
    v = adaptive_simpson(counted(f), 0.0, 1.0, tol)
    ad = calls
    n = 2
    while abs(simpson(np.sqrt, 0, 1, n) - 2 / 3) > tol: n *= 2
    print(f"tol {tol:.0e}: adaptive {ad:5d} evaluations, error {abs(v - 2/3):.1e};  uniform Simpson needs {n + 1} points")

The function $\sqrt x$ has an infinite derivative at $0$, so Simpson's $n^{-4}$ rate collapses to about $n^{-3/2}$ for a uniform grid, which then needs a million points for $10^{-10}$. Adaptive quadrature spends its evaluations where the function is hard, near $0$, and needs under a thousand. The factor $15$ comes from Simpson's error being divided by $16$ when the step is halved, and the correction $\frac{S_2-S_1}{15}$ is one step of Richardson extrapolation. Production codes (QUADPACK, used by most scientific libraries) are refined versions of this idea.

### P4.6

In [ ]:
rng = np.random.default_rng(1)
exact = 1.0                       # integral over [0,1]^d of prod (pi/2) sin(pi x_i)
for d in (1, 2, 5, 10):
    f = lambda X: np.prod(np.pi / 2 * np.sin(np.pi * X), axis=1)
    N = 10**5
    mc = f(rng.random((N, d))).mean()
    m = max(1, int(round(N ** (1 / d))))       # grid with about N points
    g = (np.arange(m) + 0.5) / m
    grid = (np.pi / 2 * np.sin(np.pi * g)).mean() ** d
    print(f"d={d:2d}: Monte Carlo error {abs(mc - exact):.1e}   midpoint grid with {m}^{d}={m**d} points error {abs(grid - exact):.1e}")

A grid with $N$ points has only $N^{1/d}$ points per axis, so its error is about $N^{-2/d}$ for the midpoint rule: excellent in dimension $1$, useless in dimension $10$, where the grid has $3$ points per axis. The Monte Carlo error is about $\sigma/\sqrt N$ in every dimension, by the central limit theorem of Volume 3. This is why high-dimensional integrals (in physics, finance, Bayesian statistics, machine learning) are computed by random sampling.